## Building A Chatbot
In this session We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This session tutorial will cover the basics which will be helpful for those two more advanced topics.

In [16]:
from langchain_core.tools import convert
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

currentkey = os.getenv("GROQ_API_KEY")
currentkey

'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'

In [24]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b",groq_api_key=currentkey)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x161a1b020>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x161a19af0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [25]:
if not currentkey:
    print("Error: API Key is None. load_dotenv() did not find the key.")
else:
    print("Key loaded successfully! (Starts with:", currentkey[:4] + "...)")

Key loaded successfully! (Starts with: gsk_...)


In [26]:
from langchain_core.messages import HumanMessage
response = model.invoke([HumanMessage(content="Hello, My name is Rasheed Ahmad")])

In [27]:
print(response)

content='Hello, Rasheed! It’s nice to meet you. How can I help you today?' additional_kwargs={'reasoning_content': 'User: "Hello, My name is Rasheed Ahmad". They greet. We should respond politely.'} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 79, 'total_tokens': 128, 'completion_time': 0.059199486, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.004435534, 'prompt_tokens_details': None, 'queue_time': 0.191102772, 'total_time': 0.06363502}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a04652-5a07-7e92-a97f-08208b866db7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 79, 'output_tokens': 49, 'total_tokens': 128, 'output_token_details': {'reasoning': 21}}


In [28]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hello, My name is Rasheed Ahmad"),
        AIMessage(content="Hello, Rasheed! It’s nice to meet you. How can I help you today?"),
        HumanMessage(content="Hello, What's my name?")
    ]
)

AIMessage(content='You’re Rasheed Ahmad.', additional_kwargs={'reasoning_content': 'The user says: "Hello, My name is Rasheed Ahmad". Assistant responded with a greeting. Now user says "Hello, What\'s my name?" This is a trick: The user already gave the name. The assistant should recall that. So answer: "Your name is Rasheed Ahmad." The user might want confirmation. So we respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 114, 'total_tokens': 200, 'completion_time': 0.09468445, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.005429369, 'prompt_tokens_details': None, 'queue_time': 0.331215178, 'total_time': 0.100113819}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04655-3f49-7b00-8c71-8aee6f818b34-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [43]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_messages_history = RunnableWithMessageHistory(model,get_session_history)

In [44]:
config = {"configurable":{"session_id":"Chat_1"}}

In [45]:
response = with_messages_history.invoke(
    [HumanMessage(content="Hi, My name is Rasheed Ahmad and i AI Engineer")],
    config = config
)

In [46]:
response.content

'Hello Rasheed! 👋 It’s great to meet an AI engineer. How can I help you today? Are you working on a particular project, looking for resources, or just curious about something in the AI space? Let me know what’s on your mind!'

In [47]:
with_messages_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)

AIMessage(content='Your name is **Rasheed Ahmad**.', additional_kwargs={'reasoning_content': 'The user says "My name is Rasheed Ahmad". They ask "What is my name?" So answer: Rasheed Ahmad.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 151, 'total_tokens': 197, 'completion_time': 0.063753146, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.048139939, 'prompt_tokens_details': None, 'queue_time': 0.195279269, 'total_time': 0.111893085}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_2b688e7cc3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0466e-2316-7ac1-b334-7e23c1286af8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 46, 'total_tokens': 197, 'output_token_details': {'reasoning': 27}})

In [49]:
config1 = {"configurable":{"session_id":"Chat2"}}
with_messages_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)

AIMessage(content='I’m not sure what your name is—could you let me know?', additional_kwargs={'reasoning_content': 'We are asked: "What is my name?" The user didn\'t provide their name. We have no context. According to policy, we should ask for clarification. We cannot guess. So we should respond: "I don\'t know your name; could you tell me?"'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 76, 'total_tokens': 154, 'completion_time': 0.080391311, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.003710952, 'prompt_tokens_details': None, 'queue_time': 0.002987994, 'total_time': 0.084102263}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0466f-5359-72a1-9621-4f32ae0d5ff2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 78, 'total_tokens': 154, 'outpu